In [1]:
import polars as pl

# Load lazily to reduce RAM usage
df = pl.read_parquet("../data/malaysia_transactions.parquet")

In [2]:
columns_to_keep = [
    "date_time",
    "ofi_entity_id",
    "rfi_entity_id",
    "trxn_amount",
    "trxn_type",
    "trxn_channel"
]

df_clean = df.select(columns_to_keep)

df_clean.null_count()

date_time,ofi_entity_id,rfi_entity_id,trxn_amount,trxn_type,trxn_channel
u32,u32,u32,u32,u32,u32
0,438281,464000,0,665434,0


In [3]:
# Drop any rows with missing sender/receiver IDs
df_filtered = df_clean.filter(
    pl.col("ofi_entity_id").is_not_null() & 
    pl.col("rfi_entity_id").is_not_null()
)

# Fill missing `trxn_type` with fallback label
df_filtered = df_filtered.with_columns(
    pl.col("trxn_type").fill_null("Unknown")
)

df_filtered.null_count()

date_time,ofi_entity_id,rfi_entity_id,trxn_amount,trxn_type,trxn_channel
u32,u32,u32,u32,u32,u32
0,0,0,0,0,0


In [4]:
df_filtered.shape

(11396551, 6)

In [5]:
# Sort the dataframe by time
df_sorted = df_filtered.sort("date_time")

# check earliest and latest timestamp
print("Start:", df_sorted["date_time"][0])
print("End:", df_sorted["date_time"][-1])

Start: 2025-06-01 00:00:00
End: 2025-06-30 23:59:59


In [6]:
from collections import defaultdict
from datetime import datetime
import numpy as np
import math

# Config
EMA_ALPHA = 0.2
DECAY_FACTOR = 0.9
MIN_HISTORY = 5

# Per-entity tracking
entity_time_profile = defaultdict(lambda: defaultdict(int))
entity_amount_profile = defaultdict(list)
entity_last_seen_day = defaultdict(lambda: None)

def update_entity_behavior(entity_id, timestamp, amount):
    hour = timestamp.hour
    day = timestamp.date()

    # Update time profile with decay
    last_day = entity_last_seen_day[entity_id]
    if last_day is not None and last_day != day:
        for h in entity_time_profile[entity_id]:
            entity_time_profile[entity_id][h] *= DECAY_FACTOR
    entity_time_profile[entity_id][hour] += 1
    entity_last_seen_day[entity_id] = day

    # Update amount history with capped length
    entity_amount_profile[entity_id].append(amount)
    if len(entity_amount_profile[entity_id]) > 50:
        entity_amount_profile[entity_id].pop(0)

def get_time_risk(entity_id, timestamp):
    hour = timestamp.hour
    histogram = entity_time_profile[entity_id]

    total = sum(histogram.values()) + 24  # smoothing
    count = histogram.get(hour, 0) + 1
    prob = count / total
    return round(1 - prob, 4)

def get_amount_risk(entity_id, amount):
    values = entity_amount_profile[entity_id]
    if len(values) < MIN_HISTORY:
        return 0.3

    avg = sum(values) / len(values)
    std = math.sqrt(sum((x - avg) ** 2 for x in values) / len(values))

    if std == 0:
        return 1.0 if amount != avg else 0.0  # exact match = safe, else very suspicious

    z = abs(amount - avg) / std
    risk = 1 - (1 / (1 + math.exp(-(z - 1.5))))  # shifted sigmoid
    return round(risk, 4)

In [7]:
eid = "E500"

# Simulate normal behavior — insert 10 values around RM 200
for i in range(10):
    ts = datetime(2025, 6, 1, 10 + i, 0, 0)
    update_entity_behavior(eid, ts, 200)

# Sanity check: show avg and std
values = entity_amount_profile[eid]
avg = sum(values) / len(values)
std = math.sqrt(sum((x - avg) ** 2 for x in values) / len(values))
print(f"Avg: {avg}, Std: {std}")

# Now test abnormal amount
test_time = datetime(2025, 6, 2, 3, 0, 0)
test_amount = 5000

amount_risk = get_amount_risk(eid, test_amount)
time_risk = get_time_risk(eid, test_time)

print("Time risk:", time_risk)
print("Amount risk:", amount_risk)

Avg: 200.0, Std: 0.0
Time risk: 0.9706
Amount risk: 1.0
